# 01 - Eksploracija Podataka (Data Exploration)

**Cilj**: Učitati, istražiti i razumeti Bitcoin Twitter dataset

**Koraci**:
1. Učitavanje dataseta
2. Osnovne statistike
3. Eksplorativna analiza (EDA)
4. Vizualizacije

In [ ]:
# Importovanje biblioteka
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Dodaj src folder u path
import sys
sys.path.append('../src')

from utils import load_twitter_data, set_style, print_dataset_info

# Postavi stil
set_style()

print("Biblioteke učitane uspešno!")

## 1. Učitavanje Podataka

**NAPOMENA**: Pre nego što pokreneš ovaj notebook, preuzmi Bitcoin Twitter dataset sa Kaggle i sačuvaj ga u `../data/raw/` folder.

Preporučeni dataset-i:
- [Bitcoin Tweets](https://www.kaggle.com/datasets/kaushiksuresh147/bitcoin-tweets)
- [Cryptocurrency Tweets](https://www.kaggle.com/datasets/alaix14/bitcoin-tweets-20160101-to-20190329)

**IZMENI putanju ispod da odgovara tvom datasetu!**

In [ ]:
# Učitaj dataset - IZMENI PUTANJU!
DATA_PATH = '../data/raw/bitcoin_tweets.csv'  # <-- IZMENI OVO!

df = load_twitter_data(DATA_PATH)

In [ ]:
# Prikaži osnovne informacije
print_dataset_info(df)

In [ ]:
# Prikaži prvih nekoliko redova
df.head(10)

## 2. Provera Kolona

**NAPOMENA**: Različiti dataset-i imaju različite nazive kolona. Potrebno je da identifikuješ:
- Kolonu sa tekstom tvita
- Kolonu sa sentiment labelom (ako postoji)
- Ostale korisne kolone (datum, likes, retweets, itd.)

In [ ]:
# Provera kolona
print("Kolone u datasetu:")
print(df.columns.tolist())

# NAKON ŠTO VIDIŠ NAZIVE KOLONA, POSTAVI IH OVDE:
TEXT_COLUMN = 'text'  # <-- Naziv kolone sa tekstom
SENTIMENT_COLUMN = 'sentiment'  # <-- Naziv kolone sa sentimentom (ako postoji)

## 3. Provera Nedostajućih Vrednosti

In [ ]:
# Nedostajuće vrednosti
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Kolona': missing_data.index,
    'Broj nedostajućih': missing_data.values,
    'Procenat': missing_percent.values
})

missing_df = missing_df[missing_df['Broj nedostajućih'] > 0].sort_values('Broj nedostajućih', ascending=False)

if len(missing_df) > 0:
    print("\n⚠️  Postoje nedostajuće vrednosti:\n")
    print(missing_df.to_string(index=False))
else:
    print("\n✅ Nema nedostajućih vrednosti!")

In [ ]:
# Ukloni redove sa nedostajućim tekstom
df = df.dropna(subset=[TEXT_COLUMN])
print(f"Nakon uklanjanja nedostajućih: {len(df)} redova")

## 4. Distribucija Sentimenta

**NAPOMENA**: Ako dataset NEMA sentiment labele, moraćeš ih ručno kreirati ili koristiti neki već labelovani dataset.

In [ ]:
# Proveri da li postoji sentiment kolona
if SENTIMENT_COLUMN in df.columns:
    print("✅ Dataset ima sentiment labele!\n")
    
    # Distribucija
    sentiment_counts = df[SENTIMENT_COLUMN].value_counts()
    print("Distribucija sentimenta:")
    print(sentiment_counts)
    print(f"\nProcenat:")
    print((sentiment_counts / len(df) * 100).round(2))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sentiment_counts.plot(kind='bar', color=['#e74c3c', '#95a5a6', '#2ecc71'], ax=ax)
    ax.set_title('Distribucija Sentimenta', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sentiment', fontsize=12)
    ax.set_ylabel('Broj tvitova', fontsize=12)
    ax.set_xticklabels(sentiment_counts.index, rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Dataset NEMA sentiment labele!")
    print("Opcije:")
    print("1. Koristi drugi dataset koji ima labele")
    print("2. Ručno labuj podatke")
    print("3. Koristi pretreniran model za labeling")

## 5. Analiza Dužine Teksta

In [ ]:
# Dužina tekstova
df['text_length'] = df[TEXT_COLUMN].apply(len)
df['word_count'] = df[TEXT_COLUMN].apply(lambda x: len(str(x).split()))

print("Statistika dužine teksta:")
print(df[['text_length', 'word_count']].describe())

In [ ]:
# Plot distribucija
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Broj karaktera
axes[0].hist(df['text_length'], bins=50, color='#3498db', edgecolor='black')
axes[0].set_title('Distribucija Broja Karaktera', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Broj karaktera', fontsize=10)
axes[0].set_ylabel('Frekvencija', fontsize=10)
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f"Mean: {df['text_length'].mean():.0f}")
axes[0].legend()

# Broj reči
axes[1].hist(df['word_count'], bins=50, color='#2ecc71', edgecolor='black')
axes[1].set_title('Distribucija Broja Reči', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Broj reči', fontsize=10)
axes[1].set_ylabel('Frekvencija', fontsize=10)
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['word_count'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Najčešće Reči (Word Cloud)

In [ ]:
# Kreiraj jedan veliki string od svih tvitova
all_text = ' '.join(df[TEXT_COLUMN].astype(str))

# Word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', 
                      colormap='viridis', max_words=100).generate(all_text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Najčešće Reči u Bitcoin Tvitovima', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 7. Primeri Tvitova

In [ ]:
# Random primeri
print("\n=== RANDOM PRIMERI TVITOVA ===")
for i, row in df.sample(5).iterrows():
    print(f"\nTvit {i}:")
    print(f"Tekst: {row[TEXT_COLUMN][:200]}..." if len(row[TEXT_COLUMN]) > 200 else f"Tekst: {row[TEXT_COLUMN]}")
    if SENTIMENT_COLUMN in df.columns:
        print(f"Sentiment: {row[SENTIMENT_COLUMN]}")
    print("-" * 80)

## 8. Čuvanje Prozornih Podataka

In [ ]:
# Sačuvaj prozoren dataset (bez duplikata, bez nedostajućih vrednosti)
df_clean = df.drop_duplicates(subset=[TEXT_COLUMN])
df_clean = df_clean.dropna(subset=[TEXT_COLUMN])

print(f"Originalan broj redova: {len(df)}")
print(f"Nakon čišćenja: {len(df_clean)}")

# Sačuvaj
output_path = '../data/processed/bitcoin_tweets_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"\n✅ Sačuvano u: {output_path}")

## 📝 Zaključak Notebook-a 01

**Urađeno**:
- ✅ Učitan Bitcoin Twitter dataset
- ✅ Izvršena osnovna EDA
- ✅ Analizirana distribucija sentimenta (ako postoji)
- ✅ Vizualizovane najčešće reči
- ✅ Očišćeni podaci sačuvani

**Sledeći korak**: Notebook 02 - Pretprocesiranje teksta